In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.datasets as datasets
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
from torch.optim.lr_scheduler import StepLR
from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter

from typing import Dict, Tuple
from tqdm import tqdm
import numpy as np
import time
import os
import random
from tabulate import tabulate

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

%matplotlib inline

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")



Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [13]:
from waveguide_dataset_paired import WaveguideDatasetPaired
dataset = WaveguideDatasetPaired('train_test_split.h5')

In [14]:
class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

In [15]:
class Net4_Mode0Weight0_original(nn.Module):
    """
    Net4 variant that predicts:
    - mode 0  (original index 0)
    - weight0 (original index 4)

    Output: [B, 2]
    Uses GroupNorm instead of BatchNorm.
    """
    def __init__(self):
        super().__init__()

        def groupnorm_2d(channels, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=channels)

        def groupnorm_1d(features, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=features)

        # ---------- CNN trunk ----------
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            groupnorm_2d(64), nn.GELU(),

            nn.Conv2d(64, 128, 3, 1, 1),
            groupnorm_2d(128), nn.GELU(),

            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(128, 256, 3, 1, 1),
            groupnorm_2d(256), nn.GELU(),

            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(256, 512, 3, 1, 1),
            groupnorm_2d(512), nn.GELU(),

            nn.MaxPool2d(2), nn.Dropout(0.25),

            Flatten()  # -> [B, 8192]
        )

        # ---------- Fully-connected head ----------
        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048),
            groupnorm_1d(2048), nn.GELU(), nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            groupnorm_1d(1024), nn.GELU(), nn.Dropout(0.3),

            nn.Linear(1024, 256),
            groupnorm_1d(256), nn.GELU(), nn.Dropout(0.25),

            nn.Linear(256, 2)
        )

    def forward(self, x_img, x_cond):
        x = self.cnn(x_img)                  # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)    # [B, 8196]
        return self.fc(x)                    # [B, 2]


In [16]:
import torch
import torch.nn as nn

class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

class Net4_Mode0Weight0_ten_layers(nn.Module):
    """
    Net4 variant that predicts:
    - mode 0  (original index 0)
    - weight0 (original index 4)

    Output: [B, 2]
    Uses GroupNorm instead of BatchNorm.
    """
    def __init__(self):
        super().__init__()

        def groupnorm_2d(channels, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=channels)

        def groupnorm_1d(features, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=features)

        # ---------- CNN trunk (10 conv layers) ----------
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),       # 1
            groupnorm_2d(64), nn.GELU(),

            nn.Conv2d(64, 64, 3, 1, 1),      # 2
            groupnorm_2d(64), nn.GELU(),

            nn.Conv2d(64, 128, 3, 1, 1),     # 3
            groupnorm_2d(128), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(128, 128, 3, 1, 1),    # 4
            groupnorm_2d(128), nn.GELU(),

            nn.Conv2d(128, 256, 3, 1, 1),    # 5
            groupnorm_2d(256), nn.GELU(),

            nn.Conv2d(256, 256, 3, 1, 1),    # 6
            groupnorm_2d(256), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(256, 512, 3, 1, 1),    # 7
            groupnorm_2d(512), nn.GELU(),

            nn.Conv2d(512, 512, 3, 1, 1),    # 8
            groupnorm_2d(512), nn.GELU(),

            nn.Conv2d(512, 512, 3, 1, 1),    # 9
            groupnorm_2d(512), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),

            nn.Conv2d(512, 512, 3, 1, 1),    # 10
            groupnorm_2d(512), nn.GELU(),

            Flatten()  # → [B, 512*4*4] = [B, 8192]
        )

        # ---------- Fully-connected head ----------
        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048),
            groupnorm_1d(2048), nn.GELU(), nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            groupnorm_1d(1024), nn.GELU(), nn.Dropout(0.3),

            nn.Linear(1024, 256),
            groupnorm_1d(256), nn.GELU(), nn.Dropout(0.25),

            nn.Linear(256, 2)
        )

    def forward(self, x_img, x_cond):
        x = self.cnn(x_img)                  # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)    # [B, 8196]
        return self.fc(x)                    # [B, 2]


In [17]:
import torch
import torch.nn as nn

class Flatten(nn.Module):
    def forward(self, x):
        return torch.flatten(x, 1)

class ResidualBlockGN(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.gn1   = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.gelu1 = nn.GELU()

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.gn2   = nn.GroupNorm(num_groups=8, num_channels=out_channels)

        self.downsample = None
        if downsample or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )

        self.activation = nn.GELU()

    def forward(self, x):
        identity = x

        out = self.gelu1(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))

        if self.downsample:
            identity = self.downsample(identity)

        out += identity
        return self.activation(out)

class Net4_Mode0Weight0(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: [B, 1, 32, 32]
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(8, 64),
            nn.GELU()
        )

        # ---------- Residual Blocks (10 conv layers total = 5 residual blocks) ----------
        self.res_blocks = nn.Sequential(
            ResidualBlockGN(64, 64),                # Conv 1–2
            ResidualBlockGN(64, 128, downsample=True),  # Conv 3–4, [B, 128, 16, 16]
            nn.Dropout(0.25),

            ResidualBlockGN(128, 256, downsample=True), # Conv 5–6, [B, 256, 8, 8]
            nn.Dropout(0.25),

            ResidualBlockGN(256, 512, downsample=True), # Conv 7–8, [B, 512, 4, 4]
            nn.Dropout(0.25),

            ResidualBlockGN(512, 512),              # Conv 9–10
        )

        self.flatten = Flatten()  # Output shape: [B, 512*4*4] = [B, 8192]

        # ---------- Fully Connected Head ----------
        def groupnorm_1d(features, num_groups=8):
            return nn.GroupNorm(num_groups=num_groups, num_channels=features)

        self.fc = nn.Sequential(
            nn.Linear(8192 + 4, 2048),
            groupnorm_1d(2048), nn.GELU(), nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            groupnorm_1d(1024), nn.GELU(), nn.Dropout(0.3),

            nn.Linear(1024, 256),
            groupnorm_1d(256), nn.GELU(), nn.Dropout(0.25),

            nn.Linear(256, 2)
        )

    def forward(self, x_img, x_cond):
        print("x_cond shape in forward:", x_cond.shape)
        x = self.stem(x_img)           # [B, 64, 32, 32]
        x = self.res_blocks(x)         # [B, 512, 4, 4]
        x = self.flatten(x)            # [B, 8192]
        x = torch.cat((x, x_cond), dim=1)  # [B, 8196]
        return self.fc(x)              # [B, 2]


In [18]:
def train_structured(model, device, loader, optimizer, loss_fn):
    """
    Train for one epoch on structured [4,2] mode-weight inputs.

    · model takes:   waveguide, params
    · model outputs: [mode0, weight0]
    · target:        cond[0] (mode0, weight0), extracted from the [4,2] tensor
    """
    model.train()

    for cond, params, waveguide in loader:
        cond, params, waveguide = (
            cond.to(device),       # [B, 4, 2]
            params.to(device),     # [B, 4]
            waveguide.to(device),  # [B, 1, 32, 32]
        )

        # ✂ Extract mode0 and weight0 as target from cond[:, 0]
        y_true = cond[:, 0, :]  # shape [B, 2]

        # forward pass
        optimizer.zero_grad()
        y_pred = model(waveguide, params)  # shape [B, 2]
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()

In [19]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch

def test_structured(model, device, loader, loss_fn, dataset, epoch_num, total_epochs):
    """
    Evaluate model for one epoch using [B, 4, 2] structured mode-weight targets.

    * Model outputs:   [B, 2] (mode-0, weight-0)
    * Ground truth:    [B, 4, 2]; we slice [0] to get first mode-weight pair

    Returns
    -------
    float
        Average MSE loss over the test set.
    """
    model.eval()
    total_loss = 0.0
    collected  = []

    # scalar stats for inverse normalization
    mode_mean_log = float(dataset.meanm[0])
    mode_std_log  = float(dataset.stdm[0])
    wt_mean_log   = float(dataset.meanw[0])
    wt_std_log    = float(dataset.stdw[0])

    with torch.no_grad():
        for cond, params, waveguide in loader:
            cond, params, waveguide = (
                cond.to(device),       # [B, 4, 2]
                params.to(device),     # [B, 4]
                waveguide.to(device),  # [B, 1, 32, 32]
            )

            # Slice first mode-weight pair as target
            y_true = cond[:, 0, :]  # [B, 2]

            # Predict
            y_pred = model(waveguide, params)  # [B, 2]
            batch_loss = loss_fn(y_pred, y_true).item()
            total_loss += batch_loss * waveguide.size(0)

            # Collect samples for plotting
            for t, o, p in zip(y_true.cpu(), y_pred.cpu(), params.cpu()):
                if len(collected) < 50:
                    # Denormalize each value
                    t_mode0_log = t[0].item() * mode_std_log + mode_mean_log
                    t_weight0_log = t[1].item() * wt_std_log + wt_mean_log
                    o_mode0_log = o[0].item() * mode_std_log + mode_mean_log
                    o_weight0_log = o[1].item() * wt_std_log + wt_mean_log

                    t_mode0 = np.expm1(t_mode0_log)
                    t_weight0 = np.expm1(t_weight0_log)
                    o_mode0 = np.expm1(o_mode0_log)
                    o_weight0 = np.expm1(o_weight0_log)

                    collected.append((
                        np.array([t_mode0, t_weight0]),
                        np.array([o_mode0, o_weight0]),
                        p.numpy()
                    ))

    avg_loss = total_loss / len(loader.dataset)

    # Plot on last epoch
    if epoch_num == total_epochs - 1 and collected:
        chosen = random.sample(collected, 8)

        fig, (ax_mode, ax_weight) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

        for tgt, out, prm in chosen:
            ax_mode.plot([0], [tgt[0]], 'ro')   # target
            ax_mode.plot([0], [out[0]], 'bx')   # output
            ax_weight.plot([0], [tgt[1]], 'ro')
            ax_weight.plot([0], [out[1]], 'bx')

        ax_mode.set_title("Mode-0")
        ax_weight.set_title("Weight-0")
        for ax in (ax_mode, ax_weight):
            ax.set_xticks([0])
            ax.set_xticklabels(['value'])
            ax.grid(True)

        plt.suptitle("Targets (red) vs Outputs (blue) on last epoch")
        plt.tight_layout()
        plt.show()

    return avg_loss


In [20]:
def main(dataset):
    os.makedirs("models", exist_ok=True)
    
    batch_size = 512
    test_batch_size = 1000
    lr = 1e-3
    gamma = 0.9
    epochs = 200
    save_dir = 'bruh_test'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = Net4_Mode0Weight0().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.MSELoss()

    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=8)

    e_loss_graph = []

    # One progress bar over all epochs
    pbar = tqdm(range(epochs), desc="Training, loss= ----")

    for epoch in pbar:
        train_structured(model, device, train_loader, optimizer, loss_fn)
        e_loss = test_structured(model, device, test_loader, loss_fn, dataset, epoch, epochs)
        e_loss_graph.append(e_loss)
        pbar.set_description()
        scheduler.step()
        torch.save(model.state_dict(), f'models/{save_dir}.pth')
        pbar.set_description(f"Training, loss: {e_loss:.4f}")
        # Save loss graph after each epoch
        plt.figure()
        plt.plot(range(epoch + 1), e_loss_graph)
        plt.title(f"Epoch loss for {save_dir}")
        plt.xlabel("Epoch")
        plt.ylabel("Test Loss")
        plt.grid(True)
        plt.savefig(f"models/{save_dir}.png")
        plt.close()
if __name__ == '__main__':
    main(dataset)

Training, loss= ----:   0%|          | 0/200 [00:00<?, ?it/s]

x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Size([512, 4])
x_cond shape in forward: torch.Siz

Training, loss= ----:   0%|          | 0/200 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# from torch.utils.data import DataLoader, random_split
# from torch.cuda.amp import GradScaler
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# import os

# def main_fast(dataset):
#     os.makedirs("models", exist_ok=True)

#     batch_size = 256
#     test_batch_size = 1000
#     lr = 1e-3
#     gamma = 0.9
#     epochs = 200
#     save_dir = 'only_top_mode4x2real'
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'

#     model = Net4_Mode0Weight0().to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
#     loss_fn = nn.MSELoss()
#     scaler = torch.amp.GradScaler()  # Explicitly specify device

#     train_size = int(0.8 * len(dataset))
#     test_size = len(dataset) - train_size
#     train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
#     test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=0)

#     e_loss_graph = []
#     pbar = tqdm(range(epochs), desc="Training, loss= ----")

#     for epoch in pbar:
#         train_structured_fast(model, device, train_loader, optimizer, loss_fn, scaler)  # AMP-enabled
#         e_loss = test_structured(model, device, test_loader, loss_fn, dataset, epoch, epochs)
#         e_loss_graph.append(e_loss)
#         scheduler.step()

#         torch.save(model.state_dict(), f'models/{save_dir}.pth')
#         pbar.set_description(f"Training, loss: {e_loss:.4f}")

#         # Save loss plot after each epoch
#         plt.figure()
#         plt.plot(range(epoch + 1), e_loss_graph)
#         plt.title(f"Epoch loss for {save_dir}")
#         plt.xlabel("Epoch")
#         plt.ylabel("Test Loss")
#         plt.grid(True)
#         plt.savefig(f"models/{save_dir}.png")
#         plt.close()

# if __name__ == '__main__':
#     main_fast(dataset)
